# AdaptiveIQ model server on a free Kaggle GPU

Runs your fine-tuned Qwen 7B GGUF with llama.cpp on Kaggle's GPU and gives it a temporary public HTTPS address.
Before running: **Settings → Accelerator → GPU T4 x2** (or P100), **Settings → Internet → On**, and
**Add-ons → Secrets → add `HF_TOKEN`** (your Hugging Face READ token) and switch it on for this notebook.

Run the cells in order. Keep this notebook **Private** and do not "Save Version" with the outputs: the last cells
print the API key. The address and key change every time you start a new session.

In [ ]:
# 1. Settings (only change HF_USER if your Hugging Face username is different)
HF_USER = "banu1234"
REPO = f"{HF_USER}/adaptiveiq-qwen7b-gguf"
FILE = "qwen7b-mcq-Q4_K_M.gguf"
LLAMA_COMMIT = "3f8af15a76ac8675812e856da45454bafc90f12c"   # the llama.cpp version this notebook was checked against
WORK = "/kaggle/working"
import os, subprocess, shutil
out = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], capture_output=True, text=True)
assert out.returncode == 0 and out.stdout.strip(), "No GPU found. Settings -> Accelerator -> GPU T4 x2, then run again."
print("GPU:", out.stdout.strip())
nvcc = shutil.which("nvcc") or ("/usr/local/cuda/bin/nvcc" if os.path.exists("/usr/local/cuda/bin/nvcc") else None)
assert nvcc, "The CUDA compiler (nvcc) is missing in this Kaggle image; send this message to Claude."
os.environ["PATH"] = os.path.dirname(nvcc) + ":" + os.environ["PATH"]
print("nvcc:", nvcc)

In [ ]:
# 2. Build llama-server with CUDA (about 5 to 15 minutes; skipped if it is already built in this session)
SRC = f"{WORK}/llama.cpp"
BIN = f"{SRC}/build/bin/llama-server"
def sh(cmd):
    print("$", cmd)
    subprocess.run(cmd, shell=True, check=True)
if not os.path.exists(BIN):
    if not os.path.isdir(SRC):
        sh(f"git init -q {SRC} && cd {SRC} && git remote add origin https://github.com/ggml-org/llama.cpp "
           f"&& git fetch -q --depth 1 origin {LLAMA_COMMIT} && git checkout -q FETCH_HEAD")
    sh(f"cmake -S {SRC} -B {SRC}/build -DGGML_CUDA=ON -DCMAKE_BUILD_TYPE=Release -DLLAMA_BUILD_TESTS=OFF > {WORK}/cmake.log 2>&1 "
       f"|| (tail -40 {WORK}/cmake.log; exit 1)")
    sh(f"cmake --build {SRC}/build --target llama-server -j 4 > {WORK}/build.log 2>&1 || (tail -40 {WORK}/build.log; exit 1)")
print("built:", BIN)

In [ ]:
# 3. Download the model from your private Hugging Face repo (about 4.7 GB)
from kaggle_secrets import UserSecretsClient
from huggingface_hub import hf_hub_download
MODEL = f"{WORK}/models/{FILE}"
if not os.path.exists(MODEL):
    token = UserSecretsClient().get_secret("HF_TOKEN")
    hf_hub_download(REPO, FILE, local_dir=f"{WORK}/models", token=token)
    del token
print("model:", MODEL, round(os.path.getsize(MODEL) / 1e9, 2), "GB")

In [ ]:
# 4. Start llama-server on the GPU, protected by a new random API key
import secrets, time, urllib.request
API_KEY = secrets.token_urlsafe(32)
env = dict(os.environ, LLAMA_API_KEY=API_KEY)
server = subprocess.Popen([BIN, "-m", MODEL, "-c", "4096", "-ngl", "99", "--host", "127.0.0.1", "--port", "8080"],
                          env=env, stdout=open(f"{WORK}/server.log", "w"), stderr=subprocess.STDOUT)
def healthy():
    try:
        return urllib.request.urlopen("http://127.0.0.1:8080/health", timeout=3).status == 200
    except Exception:
        return False
for _ in range(120):
    if healthy() or server.poll() is not None:
        break
    time.sleep(5)
if not healthy():
    print(open(f"{WORK}/server.log").read()[-3000:])
    raise RuntimeError("llama-server did not start; the log is above.")
print("llama-server is up on the GPU")

In [ ]:
# 5. Public HTTPS address with a Cloudflare quick tunnel (free, no account; a new address every session)
import re
CF = f"{WORK}/cloudflared"
if not os.path.exists(CF):
    sh(f"curl -fsSL -o {CF} https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 && chmod +x {CF}")
tunnel = subprocess.Popen([CF, "tunnel", "--no-autoupdate", "--url", "http://127.0.0.1:8080"],
                          stdout=open(f"{WORK}/tunnel.log", "w"), stderr=subprocess.STDOUT)
URL = None
for _ in range(60):
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", open(f"{WORK}/tunnel.log").read())
    if m:
        URL = m.group(0); break
    time.sleep(2)
assert URL, "No tunnel address yet:\n" + open(f"{WORK}/tunnel.log").read()[-2000:]
for _ in range(30):                      # the new address can take a few seconds to work
    try:
        if urllib.request.urlopen(URL + "/health", timeout=10).status == 200:
            break
    except Exception:
        time.sleep(3)
print("Put these two lines in backend\\.env on your laptop, then restart the backend:\n")
print(f"MODEL_BASE_URL={URL}")
print(f"MODEL_API_KEY={API_KEY}")

In [ ]:
# 6. Keep this cell running while you use the website. Stop it (or close the session) when you are done:
#    that switches the GPU off and saves your weekly hours.
try:
    while True:
        alive = healthy() and server.poll() is None and tunnel.poll() is None
        print(time.strftime("%H:%M:%S"), "model OK" if alive else "PROBLEM: run cells 4 and 5 again", flush=True)
        time.sleep(60)
except KeyboardInterrupt:
    pass
finally:
    tunnel.terminate(); server.terminate()
    print("stopped: the model and the tunnel are off")